# Examine an extracted tracking scene

This notebook loads a fixed extracted cell scene and overlays the **latest Stage 8 track-stitching result at runtime**. The saved cell volumes remain unchanged, while rerunning Stage 8 and re-executing this notebook refreshes the reconstructed centers and track paths.

In [39]:
%gui qt

In [40]:
from pathlib import Path
import sys


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root from the current working directory."""
    start = (start or Path.cwd()).resolve()

    for candidate in (start, *start.parents):
        visualizer_file = (
            candidate
            / "diagnostics"
            / "tracking_scene_extraction"
            / "napari_scene_visualizer.py"
        )
        scenes_directory = candidate / "data" / "tracking_scenes"

        if visualizer_file.is_file() and scenes_directory.is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root containing both "
        "'diagnostics/tracking_scene_extraction' and "
        "'data/tracking_scenes'."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: D:\Projects\Kaggle\cell-tracking


## Load the fixed extracted scene

Change only `SCENE_CATEGORY` and `SCENE_ID` to examine another saved case.

In [41]:
from diagnostics.tracking_scene_extraction.napari_scene_visualizer import (
    load_tracking_scene,
)


SCENE_CATEGORY = "unexpected_track_loss"
SCENE_ID = "001"

SCENE_PATH = (
    PROJECT_ROOT
    / "data"
    / "tracking_scenes"
    / SCENE_CATEGORY
    / SCENE_ID
)

scene = load_tracking_scene(SCENE_PATH)

print("Category:", scene.category)
print("Scene:", scene.name)
print("Sample:", scene.sample_id)
print("Frames:", scene.frames.tolist())
print("Volume shape (T, Z, Y, X):", scene.instance_labels.shape)
print("Crop origin ZYX:", scene.crop_origin_zyx)
print("Voxel size ZYX:", scene.voxel_size_zyx)
print("Image layers:", list(scene.masked_images))

Category: unexpected_track_loss
Scene: 001
Sample: 44b6_0113de3b
Frames: [0, 1, 2, 3]
Volume shape (T, Z, Y, X): (4, 11, 42, 43)
Crop origin ZYX: (23, 73, 144)
Voxel size ZYX: (1.625, 0.40625, 0.40625)
Image layers: ['raw', 'preprocessed']


## Load the latest Stage 8 tracks

The selected cell instances are taken from `scene.json`. Their current Stage 8 rows identify the relevant track IDs. The complete paths of those track IDs are then loaded across the scene's frame interval.

For a repaired merge, multiple rows can match the same `(frame, cell_id)` because Stage 8 replaces one merged observation with separate virtual centers.

In [42]:
import json

import numpy as np
import pandas as pd
from IPython.display import display


STAGE_8_DIR = (
    PROJECT_ROOT
    / "data"
    / "sample"
    / "processed"
    / "stage_8_track_stitching"
)

TRACKS_PATH = STAGE_8_DIR / "tracks.csv"
TRACKING_METADATA_PATH = STAGE_8_DIR / "metadata.json"
if not TRACKS_PATH.is_file():
    raise FileNotFoundError(
        f"Stage 8 tracks were not found: {TRACKS_PATH}\n"
        "Run 08_track_stitching.ipynb first."
    )

tracks = pd.read_csv(TRACKS_PATH)

required_columns = {"track_id", "frame", "z", "y", "x"}
missing_columns = required_columns - set(tracks.columns)
if missing_columns:
    raise ValueError(
        "Stage 8 tracks.csv is missing required column(s): "
        + ", ".join(sorted(missing_columns))
    )

# Verify that Stage 8 was produced for the same sample as the saved scene.
stage_8_metadata = {}
if TRACKING_METADATA_PATH.is_file():
    with TRACKING_METADATA_PATH.open("r", encoding="utf-8") as file:
        stage_8_metadata = json.load(file)

    stage_8_sample = str(stage_8_metadata.get("sample_id", ""))
    if stage_8_sample and scene.sample_id and stage_8_sample != scene.sample_id:
        raise ValueError(
            "The extracted scene and Stage 8 output belong to different samples.\n"
            f"Scene sample: {scene.sample_id}\n"
            f"Stage 8 sample: {stage_8_sample}"
        )

# The extractor records which ID column was used during manual selection.
scene_cell_column = str(
    scene.metadata.get("cell_id_column", "cell_id")
)

if scene_cell_column not in tracks.columns:
    fallback_columns = [
        column for column in ("cell_id", "cell")
        if column in tracks.columns
    ]
    if not fallback_columns:
        raise ValueError(
            "Could not find the scene cell-ID column in Stage 8 tracks.csv. "
            f"Expected '{scene_cell_column}', 'cell_id', or 'cell'."
        )
    scene_cell_column = fallback_columns[0]

selected_cells = scene.metadata.get("selected_cells", {})
if not isinstance(selected_cells, dict) or not selected_cells:
    raise ValueError(
        "scene.json does not contain any selected_cells references."
    )

selected_reference_records = [
    {
        "frame": int(frame),
        "selected_cell_id": int(cell_id),
    }
    for frame, cell_ids in selected_cells.items()
    for cell_id in cell_ids
]

selected_references = (
    pd.DataFrame(selected_reference_records)
    .sort_values(["frame", "selected_cell_id"])
    .reset_index(drop=True)
)

numeric_frame = pd.to_numeric(tracks["frame"], errors="coerce")
numeric_cell_id = pd.to_numeric(
    tracks[scene_cell_column],
    errors="coerce",
)

if "source_merged_cell_id" in tracks.columns:
    numeric_source_merged_cell_id = pd.to_numeric(
        tracks["source_merged_cell_id"],
        errors="coerce",
    )
else:
    numeric_source_merged_cell_id = pd.Series(
        np.nan,
        index=tracks.index,
        dtype=float,
    )

reference_match = pd.Series(False, index=tracks.index)

for row in selected_references.itertuples(index=False):
    frame_match = numeric_frame.eq(int(row.frame))
    cell_match = numeric_cell_id.eq(int(row.selected_cell_id))
    source_merge_match = numeric_source_merged_cell_id.eq(
        int(row.selected_cell_id)
    )

    reference_match |= frame_match & (cell_match | source_merge_match)

matched_reference_rows = tracks.loc[reference_match].copy()

if matched_reference_rows.empty:
    raise ValueError(
        "None of the saved frame/cell references were found in the current "
        "Stage 8 tracks.csv."
    )

scene_track_ids = sorted(
    pd.to_numeric(
        matched_reference_rows["track_id"],
        errors="raise",
    )
    .astype(int)
    .unique()
    .tolist()
)

scene_frame_values = set(int(value) for value in scene.frames)

scene_tracks = tracks[
    pd.to_numeric(
        tracks["track_id"],
        errors="coerce",
    ).isin(scene_track_ids)
    & numeric_frame.isin(scene_frame_values)
].copy()

scene_tracks["track_id"] = pd.to_numeric(
    scene_tracks["track_id"],
    errors="raise",
).astype(int)
scene_tracks["frame"] = pd.to_numeric(
    scene_tracks["frame"],
    errors="raise",
).astype(int)

if "is_virtual_merge" in scene_tracks.columns:
    scene_tracks["is_virtual_merge"] = (
        scene_tracks["is_virtual_merge"]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes"})
    )
else:
    scene_tracks["is_virtual_merge"] = False

scene_tracks = (
    scene_tracks
    .sort_values(["track_id", "frame"])
    .reset_index(drop=True)
)

print("Stage 8 tracks:", TRACKS_PATH)
print("Scene cell-ID column:", scene_cell_column)
print("Selected frame/cell references:", len(selected_references))
print("Matched reference rows:", len(matched_reference_rows))
print("Relevant Stage 8 track IDs:", scene_track_ids)
print("Track rows inside scene frames:", len(scene_tracks))
print("Virtual merge-center rows:", int(scene_tracks["is_virtual_merge"].sum()))

diagnostic_columns = [
    column
    for column in (
        "track_id",
        "frame",
        scene_cell_column,
        "z",
        "y",
        "x",
        "is_virtual_merge",
        "merge_event_id",
        "merge_role",
        "source_track_id",
        "source_merged_cell_id",
        "observed_merged_volume",
    )
    if column in scene_tracks.columns
]

display(scene_tracks[diagnostic_columns])

Stage 8 tracks: D:\Projects\Kaggle\cell-tracking\data\sample\processed\stage_8_track_stitching\tracks.csv
Scene cell-ID column: cell_id
Selected frame/cell references: 4
Matched reference rows: 4
Relevant Stage 8 track IDs: [89, 218]
Track rows inside scene frames: 4
Virtual merge-center rows: 0


,track_id,frame,cell_id,z,y,x,is_virtual_merge,merge_event_id,merge_role,source_track_id,source_merged_cell_id,observed_merged_volume
0,89,0,90,26.046875,90.187500,163.859375,False,NaN,NaN,89,NaN,NaN
1,218,1,99,28.016502,96.475248,163.359736,False,NaN,NaN,218,NaN,NaN
2,218,2,106,29.252459,95.439344,165.245902,False,NaN,NaN,218,NaN,NaN
3,218,3,96,29.550725,95.014493,168.017391,False,NaN,NaN,218,NaN,NaN


## Add Stage 8 paths and centers to Napari

The track coordinates are converted into the saved scene's local crop coordinates and use exactly the same scale and translation as the extracted image layers.

Layers added:

- **Stage 8 | Tracks** — complete relevant paths across the saved frame interval.
- **Stage 8 | Centers** — every center used by the corrected track table.
- **Stage 8 | Virtual merge centers** — only reconstructed centers created during merge repair.

In [43]:
from typing import Any

import napari

from diagnostics.tracking_scene_extraction.napari_scene_visualizer import (
    add_scene_to_viewer,
)


STAGE_8_LAYER_PREFIX = "Stage 8 | "


def remove_stage_8_layers(viewer: Any) -> None:
    """Remove only Stage 8 overlay layers created by this notebook."""
    for layer in list(viewer.layers):
        if str(layer.name).startswith(STAGE_8_LAYER_PREFIX):
            viewer.layers.remove(layer)


def add_stage_8_tracking_overlay(
    viewer: Any,
    scene,
    scene_tracks: pd.DataFrame,
    *,
    scene_cell_column: str,
    use_original_coordinates: bool = False,
) -> dict[str, Any]:
    """Add current Stage 8 tracks and reconstructed centers to a scene."""
    remove_stage_8_layers(viewer)

    if scene_tracks.empty:
        raise ValueError("There are no Stage 8 track rows to display.")

    frame_to_local_time = {
        int(frame): int(local_time)
        for local_time, frame in enumerate(scene.frames)
    }

    plot_rows = scene_tracks.copy()
    plot_rows["scene_time"] = plot_rows["frame"].map(frame_to_local_time)

    coordinate_columns = ["z", "y", "x"]
    for column in coordinate_columns:
        plot_rows[column] = pd.to_numeric(
            plot_rows[column],
            errors="coerce",
        )

    plot_rows = plot_rows.dropna(
        subset=["scene_time", *coordinate_columns]
    ).copy()

    if plot_rows.empty:
        raise ValueError(
            "All relevant Stage 8 rows have invalid frame or center coordinates."
        )

    crop_origin = np.asarray(
        scene.crop_origin_zyx,
        dtype=float,
    )

    plot_rows[["scene_z", "scene_y", "scene_x"]] = (
        plot_rows[["z", "y", "x"]].to_numpy(dtype=float)
        - crop_origin
    )

    scale = (1.0, *scene.voxel_size_zyx)

    if use_original_coordinates:
        spatial_translate = tuple(
            origin * spacing
            for origin, spacing in zip(
                scene.crop_origin_zyx,
                scene.voxel_size_zyx,
            )
        )
    else:
        spatial_translate = (0.0, 0.0, 0.0)

    # scene_time starts at zero, just like the extracted T axis.
    translate = (
        float(scene.frames[0]),
        *spatial_translate,
    )

    track_data = plot_rows[
        [
            "track_id",
            "scene_time",
            "scene_z",
            "scene_y",
            "scene_x",
        ]
    ].to_numpy(dtype=float)

    property_columns = [
        column
        for column in (
            "track_id",
            "frame",
            scene_cell_column,
            "is_virtual_merge",
            "merge_event_id",
            "merge_role",
            "source_track_id",
            "source_merged_cell_id",
        )
        if column in plot_rows.columns
    ]

    def property_values(series: pd.Series) -> np.ndarray:
        values = series.astype(object)
        values = values.where(pd.notna(values), "")
        return values.astype(str).to_numpy()

    properties = {
        column: property_values(plot_rows[column])
        for column in property_columns
    }

    created_layers: dict[str, Any] = {}

    created_layers["tracks"] = viewer.add_tracks(
        track_data,
        name=f"{STAGE_8_LAYER_PREFIX}Tracks",
        scale=scale,
        translate=translate,
        tail_length=max(len(scene.frames) + 2, 2),
        tail_width=3,
    )

    point_data = plot_rows[
        [
            "scene_time",
            "scene_z",
            "scene_y",
            "scene_x",
        ]
    ].to_numpy(dtype=float)

    created_layers["centers"] = viewer.add_points(
        point_data,
        name=f"{STAGE_8_LAYER_PREFIX}Centers",
        scale=scale,
        translate=translate,
        size=5,
        face_color="red",
        properties=properties,
        text={
            "string": "T{track_id}",
            "size": 8,
            "color": "white",
            "anchor": "upper_left",
        },
    )

    virtual_rows = plot_rows[
        plot_rows["is_virtual_merge"].astype(bool)
    ].copy()

    if not virtual_rows.empty:
        virtual_properties = {
            column: property_values(virtual_rows[column])
            for column in property_columns
        }

        virtual_point_data = virtual_rows[
            [
                "scene_time",
                "scene_z",
                "scene_y",
                "scene_x",
            ]
        ].to_numpy(dtype=float)

        created_layers["virtual_centers"] = viewer.add_points(
            virtual_point_data,
            name=f"{STAGE_8_LAYER_PREFIX}Virtual merge centers",
            scale=scale,
            translate=translate,
            size=9,
            face_color="yellow",
            properties=virtual_properties,
            text={
                "string": "T{track_id} {merge_role}",
                "size": 9,
                "color": "white",
                "anchor": "upper_left",
            },
        )

    return created_layers

In [44]:
USE_ORIGINAL_COORDINATES = False

viewer = napari.Viewer(
    ndisplay=3,
    title=f"Tracking Scene — {scene.category}/{scene.name}",
)

scene_layers = add_scene_to_viewer(
    viewer,
    scene,
    use_original_coordinates=USE_ORIGINAL_COORDINATES,
    remove_existing=True,
)

stage_8_layers = add_stage_8_tracking_overlay(
    viewer,
    scene,
    scene_tracks,
    scene_cell_column=scene_cell_column,
    use_original_coordinates=USE_ORIGINAL_COORDINATES,
)

viewer.dims.set_current_step(0, 0)

print(
    "Loaded the fixed scene with the latest Stage 8 paths and centers.\n"
    "Move the time slider to inspect whether each reconstructed path "
    "continues through the merged interval."
)

Loaded the fixed scene with the latest Stage 8 paths and centers.
Move the time slider to inspect whether each reconstructed path continues through the merged interval.
